## MLDS Dissertation - Formative Assignment 1 
#### Author: Benjamin Warren 
#### Due: 8th June 2026

### Report Overview 

This notebook investigates how Double Machine Learning (DML) can be used to estimate causal effects in high-dimensional advertising data, motivated by a broarder research interest in causal inference. As it stands, the aim of this dissertation is to use Double Machine Learning to estimate the causal effects of different advertising strategies, and to examine whether these effects are heterogeneous across different contexts or subgroups. 

The report proceeds in three stages. Section 1 motivates the problem, showing why naive OLS fails when confounders affect treatment and outcome through complex, high-dimensional, nonlinear relationships, illustrated with a synthetic example. Section 2 introduces the Double ML methodology, explaining orthogonalisation and cross-fitting and re-applying it to the synthetic example to show bias reduction relative to OLS. Section 3 applies the same DML approach to a real-world dataset — the iPinYou real-time bidding (RTB) log — to estimate the causal effect of payprice on click, and discusses the practical challenges (encoding, dimensionality, target leakage) that arise when moving from synthetic to real data. The notebook concludes with a discussion of DML's limitations and proposes next steps for the dissertation research.

### Section 1 - Introduction 
#### Section 1.1 - Motivating Example

Machine Learning models are often good at identifying patterns in a dataset and making predictions on unseen data. However, when the goal is to understand why something happens, predictive accuracy alone is insufficient. 

As an example, take a British sports shoe manufacturer who wants to know whether to advertise at the next Olympic games. The manufacturer thinks that when team GB do well at the games, the demand for sports equipment, including sports trainers, increases. And therefore, since the manufacturer thinks GB will do well at the next Olympics, the manufacturer wants to know if there is a causal link between GB doing well in the competition, and sports shoe sales in Britain. If there is a strong causal link, the manufacturer will decide to advertise at the next Olympic games. 

 The manufacturer obtains data of UK sports shoe sales in Olympic years and computes a naive model, which doesn't account for external factors, against the success of Team GB. The results may indicate almost no relationship and the manufacturer may conclude that British Olympic success has no effect on sports shoe sales in the UK. As a result the sports shoe manufacturer decides not to advertise at the next Olympic games. 

In order to understand why this may be the wrong result, we need to introduce the idea of confounding variables. These are variables which influence both the cause and effect in a research study. For this example, take wealth inequality in Britain as a single confounding variable. An Olympian's parents may need to be wealthy to purchase equipment, send them to schools with stronger sporting prowess and to send them to compete internationally. If there are more wealthy people in a country this may produce more Olympians as a result. 

At the same time, due to a large wealth gap, it could be the case that most of the population cannot afford, or don't have the opportunities to compete in sport. As a result, during periods where wealth inequality is high, we could see Olympic success from those with the wealth, but low trainer sales due to the majority of the population not being able to regularly compete in sporting activities.  

This demonstrates an example where the confounding variable is driving the treatment variable (Olympic success) and the outcome variable (shoe sales) in opposite directions. 

If we control for wealth inequality when predicting the effect of Olympic success on trainer sales we remove the effect of the confounding variable and are left with only the true relationship of interest. The variation in wealth inequality which increases Olympic success and decreases trainer sales is removed from our estimate. Once removed, it may reveal that Olympic success has a positive effect on trainer sales and the manufacturer of the British shoe company may be inclined to reverse their decision and decide to advertise their shoe at the next Olympics. 

Accounting for confounding variables is therefore essential.

#### Section 1.2 - Is Accounting for Confounding Variables Good Enough? 

The manufacturer's example raises a natural question: if we simply include the confounding variables in a standard regression model, will we recover a reliable causal estimate of Olympic success on trainer sales? The answer is yes, but only under a set of strict assumptions:

1. The model is correctly specified. A model is correctly specified when its functional form matches the true data-generating process. This means that the researcher must know, for every confounding variable, exactly how it relates to both the treatment and the outcome. For the motivating example, wealth inequality may affect Olympic success through a complex chain of compounding advantages - access to elite coaching, nutrition, and international competition - none of these are well described by a simple linear term and writing the exact mathematical form of this relationship would be challenging. As the number of confounders increases, the requirement that every one of these relationships is correctly specified becomes increasingly unrealistic. Any misspecification will allow residual confounding to leak into the treatment effect estimate, biasing it.  

The figure below illustrates confounding variables at the top all interacting with one another in a complex fashion before interacting with the causal estimate and outcome. The aim of the figure is to illustrate that understanding the functional form of every relationship is difficult and to introduce the notion of a causal diagram which I believe will be helpful to have pictured in ones head when reading subsequent explanations. 

<div style="text-align:center;">
    <img src="many_confounders_diagram.png" width="400">
</div>

2. OLS requires that the number of observations is substantially larger than the number of parameters being estimated. As the number of confounding variables grows, the number of parameters being estimated also grows. In high-dimensional settings, OLS coefficient estimates become increasingly noisy because there is little information per parameter. 

3. Regularisation. Techniques such as LASSO or ridge regression penalise model complexity to restore feasibility in high dimensions which is necessary when the number of parameters is large. However, regularisation shrinks coefficients towards zero to reduce variance, which means it introduces bias into every estimated coefficient, including the treatment effect. 

Taken together, these assumptions define a narrow set of conditions under which standard OLS reliably recovers a causal effect. Real-world data, particularly in high-dimension settings will routinely violate one or more of them. A more robust framework is therefore needed, one that can flexibly model the nuisance paramters without imposing these restricts on the treatment effect itself.

#### Section 1.3 - An Example of where Naive OLS Breaks Down
Within the notebook, DML_Synthetic_Data_Example.ipynb, I have created a function which generates high dimensional data where the confounding variables, $X$, affect both the treatment, $D$, and outcome, $Y$ in a complex non-linear fashion. This is displayed mathematically by the following equations:
$$
Y = D \theta_0 + g_0(X) + U,    E[U | X, D] = 0
$$
$$
D = m_0(X) + V, E[V|X] = 0
$$
where $\theta_0$ is the scalar causal parameter of interest, $g_0(X)$ and $m_0(X)$ are unknown nuisance functions, and $X \in \mathbb{R}^p$ with $p$ potentially large relative to the sample size, $N$. 

$\theta_0=3$ and represents the true causal affect of $D$ on $Y$ in this synthetic environment.  

If I run a Monte Carlo simulation over the DGP and fit a standard OLS regression each time, $Y \sim D + X$, I get the following results: 

$$
\begin{array}{lcccccc}
\hline
\text{Method} &
\text{Mean Estimate} &
\text{Bias} &
\text{Monte Carlo SD} &
\text{Average SE} &
\text{RMSE} &
\text{95\% Coverage}
\\
\hline
\text{OLS} & 4.624 & 1.624 & 0.063 & 0.073 & 1.625 & 0.0 \\
\hline
\end{array}
$$

We can see that these results are biased as the mean estimate of the coefficient is $4.624$ instead of $3$.  

A critisim of this approach could be that whilst we accounted for $X$ in the model, we didn't account for interaction terms which could have been present. However, as discussed above, in a high dimensional setting it is infeasible to account for all interaction terms as this would make the problem computationally expensive or impossible depending on the scale of the data.  

A second criticism could be that here we are trying to model a non-linear complex relationship with a linear model. One could argue that we should use a flexible machine learning model to capture the non-linear relationships. The reason we cannot sub the linear OLS for a random forest, for example, is because whilst the RF may predict Y extremely well, it will give no interpretable estimate of the treatment effect, which is what we are trying to find out. There will be no interpretable coefficient and there will be no 'true value'. **The predictive forest doesn't naturally produce a treatment effect estimate.**

#### Section 1.4 - The Mathematical Failure of Naive OLS in High Dimensions

The motivating example and Section 1.2 establish the intuition for why standard OLS fails - confounding, misspecification, and the bias-variance tension of regularisation. This section aims to formalise that intuition, using the partially linear regression (PLR) model of Chernozhukov et al. (2018).  

Suppose the true data-generating process takes the same form as above in section 1.3.

The naive regression of $Y \sim D + X$ results in the estimator: 
$$
\hat{\theta}_0 = \left( \frac{1}{n} \sum_{i \in I} D_i^2 \right)^{-1}
\frac{1}{n} \sum_{i \in I}D_i(Y_i - \hat{g}_0(X_i))
$$
where $\hat{g}_0$ is some ML estimator of $g_0$. Chernozhukov et al. (2018) show that the scaled estimation error decomposes as: 

$$
\sqrt{n} (\hat{\theta}_0 - \theta_0) = \left( \frac{1}{n} \sum D_i^2 \right)^{-1}
\frac{1}{\sqrt{n}} \sum D_i U_i + 
\left( \frac{1}{n} \sum_{i \in I} D_i^2 \right)^{-1} 
\frac{1}{\sqrt{n}} 
\sum D_i (g_0 (X_i) - \hat{g}_0(X_i))
$$
The first expression is well-behaved and converges to a normal distribution under mild conditions. The second term is the regularisation bias term. 

In a high dimensional setting, any ML estimator of $g_0$ must employ regularisation in order to generalise. As a result $\hat{g}_0$ converges to $g_0$ at a slower rate than $n^{-1/2}$. Consequently, the naive estimator is not $\sqrt{N}$-consistent. This is what is observed in the empirical example. A mean estimate of $4.624$ against a true value of $3$, with zero $95%$ coverage.  

**Any estimator that regularises $\hat{g}_0$ to achieve good prediction will induce bias in $\hat{\theta}_0$ through term the second term under naive OLS.**

### Section 2 - Double Machine Learning (Double ML)
#### Section 2.1 - Introduction to Double ML 

With these issues in mind, what we need is a general framework that allows us to separate the estimation procedure of the causal parameter from that of the nuisance parameters. Additionally, we need a framework where we don't need to specify the functional form of the nuisance parameters since this task is near impossible in a real-world setting. 


A solution to estimating a causal parameter in the presence of highly dimensional data is Double Machine Learning (also referred to as Double ML or DML). The methodology is as follows: 

Take some confounders, $X$, some treatment, $D$, and some outcome $Y$. 

Model $D \sim X$ using a ML model and store the residuals. 

Model $Y \sim X$ using a ML model and store the residuals. 

Because ML models are flexible, we don't have to make any parametric assumptions about the relationship between the confounders, $X$, and the outcome, $Y$, nor between the confounders, $X$, and the treatment, $D$, in order to get the correct treatment effect (citation: Causal Inference for the Brave and True).

The residuals can be thought of as left over variation, not explained by $X$ such that the effect of $X$ has been partialled out.  

#### Section 2.2 - How does Modelling the Residuals give a Valid Causal Estimate?

Suppose the true data-generating process takes the form the same form as in section 1.4.

After modelling $D \sim X$ and $Y \sim X$ the residuals are:  
$$
D - \hat{E}[D|X]
$$
and  
$$
Y - \hat{E}[Y|X]
$$. 

These are the parts of $D$ and $Y$ that cannot be explained by $X$.The residuals are, by construction, orthogonal to $X$.  

A naive regression of $Y \sim X$ is biased because $D$ is correlated with $X$, and $X$ also affects $Y$.  

When you regress the residuals, you are regressing two quantities that have had $X$ removed. The only remaining systematic relationship between the residuals is the direct effect of $D$ on $Y$. 

The elegance of DML is that you don't need to correctly specify the functional form of $g(X)$ or $m(X)$. As long as the models predict D and Y well enough from X, i.e. the residuals are genuinely the leftover variation, the residual regression recovers the correct causal parameter.

In this result we are using the Frisch-Waugh-Lovell theorem which states that partialling out covariates via separate regressions yields the same coefficient as including them directly in a correctly-specified joint model. DML extends this logic to the non-parametric setting: because flexible ML models are used for the nuisance estimation steps, we don't need to assume linearity or correctly specify g(X) and m(X). Provided the ML fits are sufficiently accurate, the residuals genuinely reflect leftover variation, and the causal estimate from the final OLS step remains valid. 

#### Section 2.3 Cross-Fitting

Section 2.2 explains how DML recovers the causal parameter by partialling out $X$ from both $D$ and $Y$. However, even if one uses the orthogonalised score, naively using the same data to both fit the ML nuisance models and estimate $\theta_0$ introduces a second, distinct source of bias - overfitting.  

After orthogonalisation, the DML estimator solves a score that introduces a remainder term of the form: 

$$
\frac{1}{\sqrt{n}} \sum_{i \in I} V_i (\hat{g}_0(X_i) - g_0(X_i))
$$
where $V_i = D_i - m_0(X_i)$ is the structural residual form from the the treatment equation. If the same data are used to construct $\hat{g}_0$ and to form this sum, then $\hat{g}_0(X_i)$ and $V_i$ are not independent. 

Using cross-fitting results in the nusance estimator not being trained on the fold and therefore the prediction errors, $\hat{g}_0(X_i) - g_0(X_i)$ are independent of the structural residuals, $V_i$ for the same observations. This makes the remainder term mean zero conditional on the auxiliary data, and by Chebyshev's inequality it vanishes as $n \rightarrow \infin$. 

The estimates from each fold are then aggregated meaning cross-fitting therefore recovers full-sample efficiency while eliminating the overfitting bias. 

#### Section 2.4 - DML and the Synthetic Data Example

Having established both orthogonalisation and cross-fitting, we now apply DML to the same synthetic DGP from Section 1.3. To estimate $m_0(X) = \mathbb{E}[D|X]$, a random forest classifier was used. To estimate $g_0(X) = \mathbb{E}[Y|X]$, a random forest regressor was used. Both used 400 trees, with five-fold cross-fitting applied throughout. Running the same Monte Carlo simulation as in Section 1.3 gives:

$$
\begin{array}{lcccccc}
\hline
\text{Method} &
\text{Mean Estimate} &
\text{Bias} &
\text{Monte Carlo SD} &
\text{Average SE} &
\text{RMSE} &
\text{95\% Coverage}
\\
\hline
\text{DML} & 3.734 & 0.734 & 0.052 & 0.064 & 0.735 & 0.0 \\
\hline
\end{array}
$$

The bias for the DML method compared to the standard OLS method is $0.890$ less and the average standard erorr is 0.009 less highlighting improved performance under DML.

The MC simulation only used 10 iterations and took 17 minutes to run, if I used this demonstration in my final report I would use a much higher number of iterations. To predict $D$, a random forest classifier was used. To predict $Y$, a random forest regressor was used. Both used 400 estimates. If I was to use this demonstration in my final report, I would trial different ML models to get the best estimator / fine tune the models to make this argument more convincing.

### Section 3 - Practical Example  
Now that we have introduced DML using synthetic data, lets apply it to a real world scenario and see if we have anything that we could build a dissertation around. 

#### Section 3.1 - Dataset and Research Question
The iPinYou dataset is a large-scale real-time bidding (RTB) log from an online advertising exchange, containing one row per ad impression with information on the auction (payprice, bidprice, slotprice), the publisher context (domain, slotid, region, city), the user (usertag, useragent), and the outcome of interest, click (binary). Advertiser 1458's log is used here, there is a lot more data available and the current limited data suffers from being very sparse (not many clicks). However, I need to get better at running more memory intensive code as my computer keeps crashing when I try and increase the sample. 
The treatment variable, $D$, is payprice — the price the advertiser actually paid for the impression. The outcome, $Y$, is click. The confounders, $X$, include slot characteristics, domain, region, city, user agent and user tags. The research question is whether higher auction prices causally increase the probability of a click, or whether any observed association is driven by confounding (e.g. higher-value users and premium slots both attract higher prices and higher click rates). I am only using payprice here becuase it gave interesting results.

#### Section 3.2 - Preprocessing
Before DML can be applied, the categorical and high-cardinality features must be encoded. Low-cardinality features (weekday, hour, adexchange, advertiser) are one-hot encoded. High-cardinality nominal features (city, domain, slotid, region, useragent) are target-encoded. Because target encoding uses the outcome variable, it must be performed inside each cross-fitting fold using only the training portion of the fold — otherwise information from the test fold would leak into the encoding, biasing the nuisance estimates. Multi-valued usertag entries are expanded via multi-label binarisation. Identifier columns (bidid, IP, url, etc.) and constant columns (logtype, keypage, advertiser) are dropped as they carry no information or risk overfitting.

#### Section 3.3 - Applying DML  
Script: iPinYou_DML.ipynb  


As in Section 2.4, $D \sim X$ and $Y \sim X$ are modelled with random forests, with target encoding fitted within each training fold of a 5-fold cross-fitting procedure. The residuals $D_{res}$ and $Y_{res}$ are then regressed against each other via OLS to recover $\hat{\theta}_0$, the causal effect of payprice on click. 

#### Section 3.4 - Results and Interpretation  

The final OLS regression of $Y_{res}$ on $D_{res}$​ gives a coefficient of approximately $4.68 \times 10^{−6}$ ($p \approx 0.019$), indicating a small but statistically significant positive effect of payprice on click probability. Given that payprice ranges up to roughly 300 (in Chinese Fen), the full-range effect on click probability is approximately 0.0014 — non-trivial relative to the baseline click-through rate of around 0.0008, representing close to a doubling of baseline click through rate.
The Durbin-Watson statistic (≈0.42) indicates strong residual autocorrelation, expected given the temporally ordered observations. The extreme skew (≈27) and kurtosis (≈1140) reflect the binary, rare-event nature of click and perhaps should be looked into more. 

**Practical issue**: notice that DML, designed to avoid manual functional-form choices, still required substantial manual preprocessing (one-hot encoding, target encoding, dropping columns, handling multi-valued tags). This tension — between DML's promise of flexibility and the practical reality of feature engineering for real-world tabular data — could be in itself a useful point of discussion for the dissertation, since the choices made here (which columns to drop, how to encode usertag) are themselves modelling decisions that could affect $\hat{\theta}_0$.


## Conlusion  

This notebook has shown that naive OLS produces a substantially biased estimate of a causal parameter in the presence of high-dimensional, nonlinearly-confounded data (Section 1), that DML's combination of orthogonalisation and cross-fitting meaningfully reduces this bias on synthetic data (Section 2), and that the same methodology can be applied to a real advertising dataset to estimate a plausible, if small, causal effect of auction price on click probability (Section 3).
DML is not a universal fix, however, it relies on assumptions that may not hold in practice:

- Unobserved confounding: if there are confounders not captured in $X$ (e.g. unobserved user intent), the orthogonality assumption is violated and $\hat{\theta}_0$ remains biased.  

- Poor overlap / limited variation in $D$: DML requires $D$ to vary quasi-randomly conditional on $X$; if treatment assignment is near-deterministic given $X$, the residual $D−\hat{m}_0(X)$ has little signal and estimates become unstable.  

- Poor nuisance fits: if $\hat{g}_0$ or $\hat{m}_0$ fit badly (e.g. due to inadequate preprocessing or model choice), the residuals don't reflect genuine leftover variation and the bias-reduction argument breaks down.

Next steps for the dissertation:

- Increase Monte Carlo iterations and consider a wider range of ML learners (e.g. gradient boosting, LASSO) for the nuisance models, including model selection/tuning.
- Extend the synthetic DGP to include treatment-effect heterogeneity and explore Causal Forests / DML with heterogeneous effects (e.g. CATE estimation).
- Scale the iPinYou analysis to the full dataset and across multiple advertisers to assess robustness.
- Investigate sensitivity of $\hat{\theta}_0$ to preprocessing choices (target encoding scheme, dropped columns, sample size) as a form of robustness/sensitivity analysis.
- Formalise the research proposal around a specific causal question in online advertising (e.g. effect of bid price on engagement, or effect of ad placement on conversion), using the iPinYou analysis as a pilot study.
- Think about whether to include a section of the dissertation which specifically talks about where DML fails (unobserved confounding, poor overlap, bad ML fits)

-----------------

#### Further Questions:  
1. What do you think of my tone? For example, should I give my opinion. For example, "I argue causality is important because..."
2. Would I benefit from a plot in my introduction? I could do a plot of a nations wealth against olympic medals... wealthier nations like China, USA, Australia, UK would confirm my narative. Or is a plot a bit overkill for what is otherwise just an introductory point which ultimately you don't think I'll include? 
3. Is it worth adding hetereogrneity to my DGP?
4. Is fine tuning necessary for my introductory point in Section 1. i.e. getting more convincing results to motivate this research or do you think my synthetic environment won't make it into my final paper?

Final frustrating point: DML is meant to be a solution in the high dimensional space, but I am currently going through each column and doing some feature engineering... does this not defeat the entire point of this elegant solution if I am having to go through each feature and do one-hot encoding or target encoding or dropping columns which could induce overfitting or dropping columns with only one entry? All the same preprocessing still exists because machine learning models don't natively handle categorical inputs. 